## ML Workflow Practicals

Two practical steps of running a real ML project that don't fit neatly under any single algorithm: what to do about missing values, and how to actually search for hyperparameters rather than eyeballing a leaderboard.

**Diagnosing and Fixing Missing Values**

1. Identify the mechanism first, since it decides the fix. Missing Completely at Random (MCAR): missingness is independent of everything — simple imputation or deletion is safe (deletion only under <5% missing; median/mode imputation is fast but underestimates variance). Missing at Random (MAR): missingness relates to *other observed* features (e.g. missing income correlates with age) — model-based imputation (KNN Imputer, MICE, Random Forests) preserves those correlations. Missing Not at Random (MNAR): missingness depends on the unobserved value itself (e.g. high earners refusing to report income) — needs domain knowledge or an explicit missingness indicator, not blind imputation.
2. Tree-based frameworks (XGBoost, LightGBM, CatBoost) learn optimal split directions for missing values automatically during training — no explicit imputation step needed.
3. Profiling before imputing: compute missingness rate per feature (drop features above ~40–50% unless known to be predictive); build a missingness correlation matrix (binary is_missing flags) to catch joint systematic drops, like an entire skipped sub-form; test whether missingness itself correlates with the target (Little's MCAR test) — a real difference signals MAR/MNAR, not MCAR.
4. Turning missingness into signal rather than just patching over it: add a binary is_missing_x flag alongside the imputed value, so the model can distinguish an estimate from a true observation; treat missing categoricals as their own explicit "Unknown" class rather than folding into an existing one; interaction features like is_missing_income * debt_ratio can capture conditional patterns directly.
5. Monitoring in production: track missingness rate per feature against the training baseline (data drift); PSI or KS tests on incoming distributions flag a sudden spike; schema validation at ingestion catches unexpected null/NaN/empty-string formats before they reach the model.

**Hyperparameter Tuning as a Search Problem**

1. Three separate data roles: train (fit parameters), validation (compare candidates), test (final unbiased estimate, touched exactly once). Pattern A (simple holdout): train/val/test, score each candidate once on val, evaluate the winner once on test. Pattern B (k-fold CV): train+val combined/test, that pool gets K-fold split internally so each fold takes a turn as validation.
2. Given that split, tuning is a search over three axes: the search space (which hyperparameters and their ranges, e.g. log-scale for learning rate), the search strategy (grid, random, or a sequential/Bayesian method like Optuna or Hyperband that uses past trials to pick the next candidate), and the evaluation protocol (a single held-out set, or k-fold CV — more robust, costlier, useful when data is limited).
3. Procedure per candidate hyperparameter combination: split training data into K folds; train K models, each holding out one fold as validation; average the metric across folds for that candidate's CV score; repeat for every candidate the search strategy proposes; pick the best average CV score; retrain once on the full train+val set with the winning hyperparameters; evaluate that single final model once on test for the unbiased estimate.